# Leitura dos datasets e reescrita com Gemini

Este notebook carrega os arquivos `.csv` e `.json` da pasta `data`, associa cada dataset a um DataFrame diferente e disponibiliza uma funcao para reescrever noticias com uma personalidade definida via API do Gemini.

## 1) Configuração inicial

Nesta etapa, importamos bibliotecas e carregamos as variáveis de ambiente para validar as chaves das APIs.

In [1]:
from pathlib import Path
import os
import random
import re
import time
import requests

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from openai import OpenAI

In [2]:
load_dotenv()

for env_var in ["GEMINI_API_KEY", "OPENROUTER_API_KEY", "DEEPSEEK_API_KEY"]:
    if os.getenv(env_var):
        print(f"{env_var} carregada do .env com sucesso.")
    else:
        print(f"{env_var} nao encontrada. Verifique o arquivo .env.")

GEMINI_API_KEY carregada do .env com sucesso.
OPENROUTER_API_KEY carregada do .env com sucesso.
DEEPSEEK_API_KEY carregada do .env com sucesso.


## 2) Catálogo de modelos gratuitos (OpenRouter)

Aqui consultamos os modelos disponíveis no OpenRouter e listamos apenas os que terminam com `:free`.

In [3]:
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
	headers = {"Authorization": f"Bearer {api_key}"}

	resp = requests.get("https://openrouter.ai/api/v1/models", headers=headers, timeout=30)
	resp.raise_for_status()

	models = resp.json().get("data", [])
	free_models = sorted([m["id"] for m in models if ":free" in m.get("id", "")])

	print(f"Total free ativos: {len(free_models)}")
	for mid in free_models[:30]:
		print(mid)

Total free ativos: 26
arcee-ai/trinity-large-preview:free
arcee-ai/trinity-mini:free
cognitivecomputations/dolphin-mistral-24b-venice-edition:free
google/gemma-3-12b-it:free
google/gemma-3-27b-it:free
google/gemma-3-4b-it:free
google/gemma-3n-e2b-it:free
google/gemma-3n-e4b-it:free
liquid/lfm-2.5-1.2b-instruct:free
liquid/lfm-2.5-1.2b-thinking:free
meta-llama/llama-3.2-3b-instruct:free
meta-llama/llama-3.3-70b-instruct:free
minimax/minimax-m2.5:free
mistralai/mistral-small-3.1-24b-instruct:free
nousresearch/hermes-3-llama-3.1-405b:free
nvidia/nemotron-3-nano-30b-a3b:free
nvidia/nemotron-3-super-120b-a12b:free
nvidia/nemotron-nano-12b-v2-vl:free
nvidia/nemotron-nano-9b-v2:free
openai/gpt-oss-120b:free
openai/gpt-oss-20b:free
qwen/qwen3-4b:free
qwen/qwen3-coder:free
qwen/qwen3-next-80b-a3b-instruct:free
stepfun/step-3.5-flash:free
z-ai/glm-4.5-air:free


## 3) Carregamento e organização dos datasets

Nesta parte, carregamos os arquivos da pasta `data`, criamos os DataFrames e associamos nomes para facilitar os testes.

In [5]:
def read_dataset(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(file_path)
    if suffix == ".json":
        return pd.read_json(file_path)

    raise ValueError(f"Formato de arquivo nao suportado: {file_path.name}")


data_dir = Path("../data")
dataset_files = sorted(
    [*data_dir.glob("*.csv"), *data_dir.glob("*.json")],
    key=lambda path: path.name.lower(),
)

if not dataset_files:
    raise FileNotFoundError("Nenhum arquivo CSV ou JSON foi encontrado na pasta 'data'.")

dataset_files

[WindowsPath('../data/Latest_News.csv'),
 WindowsPath('../data/Latest_News.json'),
 WindowsPath('../data/Science_&_Technology_News.csv'),
 WindowsPath('../data/Science_&_Technology_News.json'),
 WindowsPath('../data/World_Politics_News.csv'),
 WindowsPath('../data/World_Politics_News.json')]

In [6]:
def to_variable_name(file_path: Path) -> str:
    name = file_path.stem.lower()
    name = re.sub(r"\W+", "_", name)
    name = re.sub(r"^(\d)", r"df_\1", name)
    return name


dataframes = {}

for dataset_file in dataset_files:
    df_name = to_variable_name(dataset_file)
    df = read_dataset(dataset_file)
    dataframes[df_name] = df
    globals()[df_name] = df

list(dataframes.keys())

['latest_news', 'science___technology_news', 'world_politics_news']

In [16]:
for df_name, df in dataframes.items():
    print(f"{df_name}: {df.shape[0]} linhas x {df.shape[1]} colunas")

latest_news: 86560 linhas x 11 colunas
science___technology_news: 5437 linhas x 11 colunas
world_politics_news: 2933 linhas x 11 colunas


## 4) Funções de processamento e reescrita

Nesta seção, o notebook define as funções utilitárias para:
- escolher o melhor campo de texto da notícia,
- montar o prompt de reescrita,
- chamar os providers (Gemini/OpenRouter/DeepSeek),
- controlar status e erros por linha no DataFrame.

In [17]:
TEXT_COLUMN_CANDIDATES = ("full_description", "content", "description", "title")
DETAILED_TEXT_COLUMNS = ("full_description", "content", "description")


def choose_news_text_column(
    df: pd.DataFrame,
    candidates=TEXT_COLUMN_CANDIDATES,
    prefer_detailed_text: bool = True,
) -> str:
    if prefer_detailed_text:
        priority_candidates = [column for column in DETAILED_TEXT_COLUMNS if column in candidates]
    else:
        priority_candidates = list(candidates)

    fallback_candidates = [column for column in candidates if column not in priority_candidates]

    for column in [*priority_candidates, *fallback_candidates]:
        if column not in df.columns:
            continue

        non_empty_count = df[column].fillna("").astype(str).str.strip().ne("").sum()
        if non_empty_count > 0:
            return column

    raise ValueError(
        "Nenhuma coluna de texto foi encontrada. Informe 'text_column' manualmente."
    )


def resolve_row_text(
    row: pd.Series,
    preferred_column: str | None = None,
    candidates=TEXT_COLUMN_CANDIDATES,
    allow_title_fallback: bool = False,
) -> tuple[str, str]:
    ordered_candidates = []
    if preferred_column:
        ordered_candidates.append(preferred_column)

    ordered_candidates.extend(column for column in candidates if column not in ordered_candidates)

    if not allow_title_fallback:
        ordered_candidates = [column for column in ordered_candidates if column != "title"]

    for column in ordered_candidates:
        if column not in row.index:
            continue

        value = row[column]
        text = "" if pd.isna(value) else str(value).strip()
        if text:
            return column, text

    if allow_title_fallback and "title" in row.index:
        title_value = row["title"]
        title_text = "" if pd.isna(title_value) else str(title_value).strip()
        if title_text:
            return "title", title_text

    raise ValueError("A linha nao possui texto utilizavel nas colunas candidatas.")


def generate_rewrite_with_retry_gemini(
    client: genai.Client,
    *,
    model: str,
    prompt: str,
    system_instruction: str,
    max_attempts: int = 5,
    base_delay: float = 2.0,
) -> str:
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = client.models.generate_content(
                model=model,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.8,
                ),
                contents=prompt,
            )

            rewritten_text = (response.text or "").strip()
            if not rewritten_text:
                raise ValueError("A resposta da API veio vazia.")

            return rewritten_text
        except Exception as exc:
            last_error = exc
            error_text = str(exc)
            is_retryable = (
                "429" in error_text
                or "RESOURCE_EXHAUSTED" in error_text
                or "503" in error_text
                or "UNAVAILABLE" in error_text
            )

            if not is_retryable or attempt == max_attempts:
                break

            sleep_for = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 1)
            time.sleep(sleep_for)

    raise last_error


def generate_rewrite_with_retry_openai_compatible(
    client: OpenAI,
    *,
    model: str,
    prompt: str,
    system_instruction: str,
    max_attempts: int = 5,
    base_delay: float = 2.0,
) -> str:
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                temperature=0.8,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": prompt},
                ],
            )

            rewritten_text = (response.choices[0].message.content or "").strip()
            if not rewritten_text:
                raise ValueError("A resposta da API veio vazia.")

            return rewritten_text
        except Exception as exc:
            last_error = exc
            error_text = str(exc)
            is_retryable = (
                "429" in error_text
                or "503" in error_text
                or "UNAVAILABLE" in error_text
                or "rate limit" in error_text.lower()
                or "timeout" in error_text.lower()
            )

            if not is_retryable or attempt == max_attempts:
                break

            sleep_for = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 1)
            time.sleep(sleep_for)

    raise last_error


def rewrite_news_with_personality(
    df: pd.DataFrame,
    personality: str,
    text_column: str | None = None,
    title_column: str = "title",
    model: str = "gemini-2.5-flash-lite",
    provider: str = "gemini",
    api_key: str | None = None,
    output_column: str = "rewritten_news",
    max_rows: int | None = None,
    sleep_seconds: float = 0.0,
    retry_attempts: int = 5,
    allow_title_fallback: bool = False,
) -> pd.DataFrame:
    if df.empty:
        raise ValueError("O DataFrame esta vazio.")

    if not personality or not personality.strip():
        raise ValueError("Informe uma personalidade valida.")

    provider_normalized = provider.strip().lower()
    if provider_normalized not in {"gemini", "openrouter", "deepseek"}:
        raise ValueError("Provider invalido. Use: gemini, openrouter ou deepseek.")

    if provider_normalized == "gemini":
        env_key_name = "GEMINI_API_KEY"
    elif provider_normalized == "openrouter":
        env_key_name = "OPENROUTER_API_KEY"
    else:
        env_key_name = "DEEPSEEK_API_KEY"

    resolved_api_key = api_key or os.getenv(env_key_name)
    if not resolved_api_key:
        raise ValueError(
            f"Defina a {env_key_name} no ambiente ou passe a chave em 'api_key'."
        )

    resolved_text_column = text_column or choose_news_text_column(df)
    if resolved_text_column not in df.columns:
        raise ValueError(f"A coluna '{resolved_text_column}' nao existe no DataFrame.")

    if provider_normalized == "gemini":
        client = genai.Client(api_key=resolved_api_key)
    elif provider_normalized == "openrouter":
        client = OpenAI(
            api_key=resolved_api_key,
            base_url="https://openrouter.ai/api/v1",
            default_headers={
                "HTTP-Referer": os.getenv("OPENROUTER_HTTP_REFERER", ""),
                "X-Title": os.getenv("OPENROUTER_X_TITLE", "misinformation-llm-simulation"),
            },
        )
    else:
        client = OpenAI(
            api_key=resolved_api_key,
            base_url="https://api.deepseek.com",
        )

    rewritten_df = df.copy()
    rewritten_df["source_text_column"] = pd.NA
    rewritten_df[output_column] = pd.NA
    rewritten_df["rewrite_status"] = "not_requested"
    rewritten_df["rewrite_error"] = pd.NA

    target_indexes = list(rewritten_df.index)
    if max_rows is not None:
        target_indexes = target_indexes[:max_rows]

    system_instruction = (
        "Voce reescreve noticias conforme a personalidade pedida, mantendo os fatos centrais "
        "e sem inventar informacoes novas."
    )

    for row_index in target_indexes:
        row = rewritten_df.loc[row_index]

        try:
            source_column, original_text = resolve_row_text(
                row=row,
                preferred_column=resolved_text_column,
                allow_title_fallback=allow_title_fallback,
            )
            rewritten_df.at[row_index, "source_text_column"] = source_column
        except ValueError as exc:
            rewritten_df.at[row_index, "rewrite_status"] = "skipped"
            rewritten_df.at[row_index, "rewrite_error"] = str(exc)
            continue

        title = ""
        if title_column in rewritten_df.columns and pd.notna(rewritten_df.at[row_index, title_column]):
            title = str(rewritten_df.at[row_index, title_column]).strip()

        prompt = f"""
Assuma a seguinte personalidade ao reescrever a noticia:
{personality}

Regras:
- Reescreva no mesmo idioma do texto original.
- Preserve os fatos centrais.
- Nao invente dados, numeros, citacoes ou personagens.
- Ajuste tom, vocabulario e estilo para refletir a personalidade pedida.
- Retorne apenas o texto reescrito.

Titulo:
{title or 'Sem titulo'}

Texto original:
{original_text}
""".strip()

        try:
            if provider_normalized == "gemini":
                rewritten_text = generate_rewrite_with_retry_gemini(
                    client,
                    model=model,
                    prompt=prompt,
                    system_instruction=system_instruction,
                    max_attempts=retry_attempts,
                )
            else:
                rewritten_text = generate_rewrite_with_retry_openai_compatible(
                    client,
                    model=model,
                    prompt=prompt,
                    system_instruction=system_instruction,
                    max_attempts=retry_attempts,
                )

            rewritten_df.at[row_index, output_column] = rewritten_text
            rewritten_df.at[row_index, "rewrite_status"] = "success"
            rewritten_df.at[row_index, "rewrite_error"] = pd.NA
        except Exception as exc:
            rewritten_df.at[row_index, "rewrite_status"] = "error"
            rewritten_df.at[row_index, "rewrite_error"] = str(exc)

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    return rewritten_df

## 5) Testes com APIs remotas

A seguir executamos reescritas com providers remotos e avaliamos os campos `rewrite_status` e `rewrite_error`.

In [29]:
science_rewritten_df = rewrite_news_with_personality(
    df=dataframes["science___technology_news"].head(10),
    personality="um jornalista sensacionalista e alarmista",
    provider="gemini",
    max_rows=5,
    sleep_seconds=5.0,
    retry_attempts=2,
    text_column="description",
    model="gemini-2.5-flash-lite",
)



In [30]:
science_rewritten_df[["title", "source_text_column", "rewritten_news", "rewrite_status", "rewrite_error"]].head()

,title,source_text_column,rewritten_news,rewrite_status,rewrite_error
0,OPPO Reno6 5G Recensione: uno smartphone elega...,description,**ALERTA MÁXIMO! O NOVO OPPO RENO6 5G CHEGOU E...,success,<NA>
1,Un artista smonta Xiaomi MIX Fold: il risultat...,description,**ESCÂNDALO EXPLOSIVO! Xiaomi MIX Fold DESMONT...,success,<NA>
2,"WHO: Staaten nicht in der Lage, Pandemie bald ...",description,**O MUNDO À BEIRA DO COLAPSO! OMS EMITE ALERTA...,success,<NA>
3,Google Pay már az Erste Banknál is,description,**PÂNICO NAS FINANÇAS! Google Pay INVADE o Ers...,success,<NA>
4,Antipasto di Black Friday su Amazon: che scont...,description,ALERTA MÁXIMO: AMAZON LANÇA OFENSIVA DO BLACK ...,success,<NA>


In [35]:
openrouter_llama_rewritten_df = rewrite_news_with_personality(
    df=dataframes["science___technology_news"].head(10),
    personality="um jornalista sensacionalista e alarmista",
    provider="openrouter",
    model="meta-llama/llama-3.3-70b-instruct:free",
    max_rows=3,
    sleep_seconds=3.0,
    retry_attempts=2,
    text_column="description",
)

openrouter_nvidia_rewritten_df = rewrite_news_with_personality(
    df=dataframes["science___technology_news"].head(10),
    personality="um jornalista sensacionalista e alarmista",
    provider="openrouter",
    model="nvidia/nemotron-3-super-120b-a12b:free",
    max_rows=3,
    sleep_seconds=3.0,
    retry_attempts=2,
    text_column="description",
)

print("OpenRouter (Llama 3.3 70B free) status:")
print(openrouter_llama_rewritten_df["rewrite_status"].value_counts(dropna=False))
print()
print("OpenRouter (Nvidia Nemotron 3 Super free) status:")
print(openrouter_nvidia_rewritten_df["rewrite_status"].value_counts(dropna=False))

for label, df_result in [
    ("OpenRouter Llama", openrouter_llama_rewritten_df),
    ("OpenRouter Nvidia", openrouter_nvidia_rewritten_df),
]:
    erros = df_result[df_result["rewrite_status"] == "error"]
    if erros.empty:
        continue

    print(f"\nErros em {label}:")
    for idx, row in erros.iterrows():
        print(f"- linha {idx}: {row['rewrite_error']}")

OpenRouter (Llama 3.3 70B free) status:
rewrite_status
not_requested    7
success          2
error            1
Name: count, dtype: int64

OpenRouter (Nvidia Nemotron 3 Super free) status:
rewrite_status
not_requested    7
success          3
Name: count, dtype: int64

Erros em OpenRouter Llama:
- linha 2: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'meta-llama/llama-3.3-70b-instruct:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Venice', 'is_byok': False}}, 'user_id': 'user_3B8tEMyUTe570OCIPbqUZesaCpH'}


## 6) Teste com modelo local (Ollama)

Nesta parte fazemos uma chamada local via endpoint compatível com OpenAI (`http://127.0.0.1:11434/v1`) para validar geração sem depender de API externa.

In [36]:
local_client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
)

sample_text = dataframes["science___technology_news"].head(1)["description"].iloc[0]

prompt = f"""
Assuma a personalidade: um jornalista sensacionalista e alarmista.
Reescreva no mesmo idioma, preserve fatos, sem inventar.
Retorne apenas o texto reescrito.

Texto original:
{sample_text}
""".strip()

response = local_client.chat.completions.create(
    model="llama3.1:8b",
    temperature=0.8,
    messages=[
        {"role": "system", "content": "Voce reescreve noticias sem inventar fatos."},
        {"role": "user", "content": prompt},
    ],
)

print(response.choices[0].message.content)

**ATENÇÃO: IL NUOVO OPPO RENO6 5G È UN DISPOSITIVO CHE STUPIRÀ TUTTI!**

Abbiamo testato l'OPPO Reno6 5G e la nostra esperienza è stata... ECCEZIONALE!

Il nuovo modello ha lasciato senza fiato i nostri redattori con il suo design unico e inconfondibile, ma non solo. Abbiamo utilizzato il dispositivo nelle ultime settimane e abbiamo scoperto...

(Ecco dove finisce la notizia reescrita)
